#**ClinRAG**

##**Knowledge Indexing**
Prepare the raw data and convert it into vectoDB
1. Document Loading
2. Chunking
3. Embedding
4. Vector Store

In [1]:
# ! pip install langchain

In [2]:
# ! pip install langchain-openai

In [3]:
# ! pip install langchain-community

In [4]:
# ! pip install langchain-chroma

###**1. Document Loading**

In [5]:
# Load the documents from the data directory
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import TextLoader

loader = DirectoryLoader('./data', glob='**/*.txt', show_progress=True, loader_cls=TextLoader)
loaded_docs = loader.load()

print(f"Loaded {len(loaded_docs)} documents.")

100%|██████████| 3/3 [00:00<00:00, 483.59it/s]

Loaded 3 documents.


###**2. Chunking**

In [6]:
# Chunk the documents into smaller pieces
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

chunked_docs = text_splitter.create_documents([doc.page_content for doc in loaded_docs])

print(f"Created {len(chunked_docs)} chunks.")

Created 181 chunks.


###**3. Embedding**

In [7]:
# Initialize the embedding model

from langchain_openai import AzureOpenAIEmbeddings

keyFile = open('keys/.azure_openai_embedding_key.txt', 'r')
AZURE_OPENAI_API_KEY = keyFile.read().strip()
keyFile.close()

embeddingModel = AzureOpenAIEmbeddings(
    azure_endpoint="https://gopi-m5ncld8s-eastus2.openai.azure.com/",
    azure_deployment="text-embedding-3-small",
    api_key=AZURE_OPENAI_API_KEY)

In [8]:
# Initialize the vector store
from langchain_chroma import Chroma

db = Chroma(collection_name='clin_rag_alzheimers',
                embedding_function=embeddingModel,
                persist_directory='./vector_store')

In [9]:
#reset the collection before adding the documents
db.reset_collection()

# Add chunked documents to the vector store
db.add_documents(chunked_docs)

print(len(db.get()["ids"]))

181
